# SHAPE-CE Analysis Notebook (`shapece`)
### From raw capillary-electrophoresis traces to reactivity and RNA structure

This notebook analyzes chemical-probing experiments (SHAPE, DMS, CMCT, hydroxyl-radical, enzymatic)
run on a capillary sequencer. It follows the analysis logic of **QuShape** (Karabiber et al., *RNA*
2013), reimplemented in modern Python 3, and adds size-standard alignment and ViennaRNA
structure prediction.

**You do not need to edit any code.** Every setting is a button, dropdown, or text box.

**How to use:** run each cell in order (`Shift`+`Enter`). Cells with a 🔧 icon show a form —
fill it in, then run the next cell. Cells with 📊 show a plot to check your data.


## Step 0 — Install
Run once. Takes about a minute on Colab.

In [ ]:
try:
    import google.colab  # noqa
    IN_COLAB = True
    !pip -q install ViennaRNA biopython openpyxl ipywidgets
    !pip -q install git+https://github.com/johnnythor-micro/shape-ce.git
except ImportError:
    IN_COLAB = False

import numpy as np, matplotlib.pyplot as plt
import shapece as sc
from shapece import ui
print("shapece", sc.__version__, "ready")

## Step 1 — 🔧 Upload your `.fsa` files

Click **Choose Files** and select every trace file for this experiment: your (+) reagent lanes,
your (−) background lanes, and any sequencing-ladder lanes.

In [ ]:
paths = ui.upload_files(accept=".fsa")
print(f"{len(paths)} file(s) ready.")

## Step 2 — 🔧 Describe your experiment

Three forms appear:

1. **RNA sequence** — paste a plain sequence or a FASTA record. Headers, line breaks, numbers and
   `T`→`U` are handled automatically; it tells you the length and GC content.
2. **Channels** — pick which detector channel holds the reagent trace and which holds the size
   standard. The dropdowns list the channels actually present in your file, with their signal
   ranges. *The reagent channel has many sharp peaks; the size standard is an evenly spaced ladder.*
   Choose **none** if your run had no size standard.
3. **Files** — set each file's **role** ((+) reagent, (−) background, ladder, ignore), its
   **condition**, and its **replicate number**. Each (+) is automatically paired with the (−)
   that shares its condition and replicate.

Fill everything in, then run the next cell.

In [ ]:
panel = ui.ConfigPanel(paths, conditions=("condition 1", "condition 2")).display()

### Confirm your setup
Run this after filling the forms above. Re-run it any time you change something.

In [ ]:
CONFIG = panel.to_config()
print(panel.summary())
assert CONFIG["pairs"], "No (+)/(-) pairs — check the roles, conditions and replicate numbers above."
assert CONFIG["rna_sequence"], "Paste your RNA sequence in the box above."


## Step 3 — 📊 Preprocess and inspect

Repairs saturated peaks, smooths, and removes baseline drift.

**What to look for:** sharp peaks preserved, baseline flat at zero, no wide clipped plateaus.

In [ ]:
from shapece.io import read_abif, get_channel
from shapece import preprocess as pp

traces, size = {}, {}
for name, path in CONFIG["samples"].items():
    e = read_abif(path)
    traces[name] = {"RX": pp.baseline(pp.smooth(pp.correct_saturation(
        get_channel(e, CONFIG["data_channel"]))))}
    if CONFIG["size_channel"]:
        size[name] = get_channel(e, CONFIG["size_channel"])

ref = CONFIG["reference"]
plt.figure(figsize=(13, 3))
plt.plot(traces[ref]["RX"], lw=0.5); plt.title(f"cleaned trace — {ref}")
plt.xlabel("scan"); plt.tight_layout(); plt.show()

## Step 4 — 📊 Align the lanes

With a size standard, each lane is warped onto a reference lane using the standard's peaks — the
most reliable method, because the standard runs in the same capillary as your data. Without one,
dynamic time warping aligns the traces directly.

**What to look for:** lanes of the same condition should lie almost on top of each other.

In [ ]:
from shapece import align

if CONFIG["size_channel"]:
    aligned = align.size_standard_align(traces, size, reference=CONFIG["reference"], roles=("RX",))
else:
    aligned = align.dtw_align(traces, reference=CONFIG["reference"], align_on="RX", roles=("RX",))

plus_lanes = [p for p, m in CONFIG["pairs"]]
plt.figure(figsize=(13, 3))
for lab in plus_lanes:
    plt.plot(aligned[lab]["RX"], lw=0.4, alpha=0.75, label=lab)
plt.legend(fontsize=7); plt.title("aligned (+) lanes — should overlap")
plt.xlabel("scan"); plt.tight_layout(); plt.show()

## Step 5 — 📊 Detect and quantify peaks

Peaks are found once on the average of the (+) lanes, then measured at those same positions in
every lane (Gaussian deconvolution), so all lanes stay directly comparable.

In [ ]:
from shapece import peaks

ref_trace = np.mean([aligned[l]["RX"] for l in plus_lanes], axis=0)
pos = peaks.detect_peaks(ref_trace, min_spacing=12, prominence_frac=0.03)
quant = {name: peaks.quantify(aligned[name]["RX"], pos, mode="gaussian") for name in traces}
print(f"{len(pos)} peaks detected")

plt.figure(figsize=(13, 3))
plt.plot(ref_trace, lw=0.5, color="0.4"); plt.plot(pos, ref_trace[pos], "rv", ms=3)
plt.title("consensus peaks"); plt.xlabel("scan"); plt.tight_layout(); plt.show()

## Step 6 — ⚠️ Did the probe actually work?

**This is the most important cell in the notebook.**

When the reagent is weak or degraded, the (+) and (−) lanes are nearly identical. Background
subtraction then cannot isolate reactivity, and what survives is the natural stop pattern —
which is *highly reproducible*, so replicates agree beautifully and the data look clean.

**A reproducible background is not a measurement.** If this check fails, repeat the experiment
with fresh reagent; no downstream analysis can rescue it.

In [ ]:
from shapece import stats

for plus, minus in CONFIG["pairs"]:
    qc = stats.probe_signal_qc(quant[plus]["area"], quant[minus]["area"])
    flag = "PASS" if qc["passed"] else "FAIL"
    print(f"[{flag}] {plus} vs {minus}: "
          f"r(+,-)={qc['rx_bg_correlation']:.3f}  "
          f"ratio={qc['mean_area_ratio']:.2f}  "
          f"carryover={qc['background_carryover']:+.3f}")
    for r in qc["reasons"]:
        print("        -", r)

## Step 7 — 📊 Compute reactivity

Background is scaled to the reagent using the least-reactive peaks, subtracted, then normalized
with the model-free boxplot rule so a reactive nucleotide ≈ 1.0.

**What to look for:** most values between 0 and 2; replicates tracking each other.

In [ ]:
from shapece import reactivity as rx
from shapece import plots

profiles = {}
for plus, minus in CONFIG["pairs"]:
    if CONFIG["reactivity_model"] == "area_difference":
        r = rx.area_difference(quant[plus]["area"], quant[minus]["area"], scale=True)
    else:
        r = np.clip(rx.stop_fraction(aligned[plus]["RX"], pos)
                    - rx.stop_fraction(aligned[minus]["RX"], pos), 0, None)
    profiles[plus], _ = rx.boxplot_normalize(r)

plots.skyline(profiles, title="reactivity profiles"); plt.tight_layout(); plt.show()

## Step 8 — Export

Writes a `.shape` file (for folding) and a GraphPad Prism–ready Excel workbook.
Files appear in the file browser on the left; in Colab, double-click to download.

In [ ]:
from shapece import report

seq = CONFIG["rna_sequence"]
nt = np.arange(1, len(pos) + 1)      # peak order; use a ladder for absolute numbering
df = report.reactivity_table(nt, ["N"] * len(nt), profiles)
report.write_excel_for_prism("reactivity_for_prism.xlsx", df)

first = list(profiles)[0]
report.write_shape(f"{first}.shape", nt, profiles[first], seq_len=len(seq))
print("wrote reactivity_for_prism.xlsx and", f"{first}.shape")
df.head()

## Step 9 — 📊 Predict and draw the structure

Folds your RNA with the reactivities as constraints (ViennaRNA, Deigan method) and draws it two ways.

**Arc plot:** pairs as arcs over the sequence. Nested arcs are helices; crossing arcs are pseudoknots.
In a good model, red (reactive) nucleotides land in the gaps between arcs.
**Circle plot:** better for long RNAs, where arcs get too tall to read.

In [ ]:
r_full = np.full(len(seq), np.nan)
for i, v in zip(nt, profiles[first]):
    if 1 <= i <= len(seq):
        r_full[i - 1] = v

ss_shape, mfe_s = sc.structure.fold(seq, r_full, method="deigan")
ss_thermo, mfe_t = sc.structure.fold(seq, None)
print(f"SHAPE-directed: {mfe_s:.1f} kcal/mol   thermodynamic: {mfe_t:.1f} kcal/mol")

fig, ax = plt.subplots(figsize=(14, 5))
plots.arc(ss_shape, reactivity=r_full, sequence=seq, structure2=ss_thermo, ax=ax,
          labels=("SHAPE-directed", "thermodynamic"))
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 8))
plots.circle(ss_shape, reactivity=r_full, sequence_length=len(seq), ax=ax)
plt.tight_layout(); plt.show()

## Next steps

- **Comparing two conditions?** Open `SHAPE_CE_Comparison.ipynb` for deltaSHAPE and statistics.
- **Statistics in Prism?** See `docs/STATISTICS_GUIDE.md`.
- **What do the plots mean?** See `docs/VISUALIZATION_GUIDE.md`.

### Prefer to edit code instead of forms?
Every widget just builds a plain dict. You can skip Steps 1–2 and write it yourself:

```python
CONFIG = {
    "rna_sequence": "GGGAUC...",
    "data_channel": 9, "size_channel": 205,
    "samples": {"plus_r1": "/path/a.fsa", "minus_r1": "/path/b.fsa"},
    "pairs": [("plus_r1", "minus_r1")],
    "reference": "plus_r1",
    "reactivity_model": "area_difference",
}
```

**Citation:** QuShape (Karabiber et al., *RNA* 2013); ViennaRNA (Lorenz et al., 2011);
Deigan et al. (2009); Low & Weeks (2010); and this repository.